# 02 — BM25 baseline & retrieval eval

**Câu hỏi của notebook này:** với 4.133 chunk và 12 query đã chấm, BM25 tìm đúng bài được bao nhiêu phần trăm?

## Nguyên tắc eval-first

Toàn bộ giá trị của project này nằm ở chỗ: **mọi thay đổi pipeline đều được đo bằng cùng một bộ qrels**.
Nên bước đầu tiên không phải là BM25 — mà là **thước đo**.

## Quy ước đã chốt (ghi rõ để nhất quán về sau)

| Khái niệm | Định nghĩa |
|---|---|
| "relevant" (nhị phân) | `relevance >= 1` — dùng cho Recall@k, Precision@k, MRR |
| "graded relevance" | điểm nguyên `0 / 1 / 2` — dùng cho nDCG@k |
| đơn vị xếp hạng | **bài viết (doc_id)**, không phải chunk — vì qrels chấm ở mức bài |

---
# PHẦN A — Viết các hàm metric

In [1]:
# ── Setup: nạp chunks + qrels (chạy 1 lần) ─────────────────────
import json, math, random
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CHUNKS = ROOT / "data" / "processed" / "chunks.jsonl"
QRELS = ROOT / "eval" / "qrels.csv"

chunks = pd.read_json(CHUNKS, lines=True)
chunks["doc_id"] = chunks["doc_id"].astype(str)

qrels = pd.read_csv(QRELS, dtype={"doc_id": str})
qrels = qrels.dropna(subset=["doc_id", "relevance"])

QUERIES = dict(zip(qrels["query_id"], qrels["query"]))            # {qid: câu hỏi}
QUERY_CLASS = dict(zip(qrels["query_id"], qrels["query_class"]))  # {qid: lớp}
GOLD_BINARY = qrels[qrels["relevance"] >= 1].groupby("query_id")["doc_id"].apply(set).to_dict()
GOLD_GRADED = qrels.groupby("query_id").apply(
    lambda g: dict(zip(g["doc_id"], g["relevance"])), include_groups=False).to_dict()

print(f"chunks : {len(chunks):,} · {chunks['doc_id'].nunique()} bài")
print(f"queries: {len(QUERIES)} · số dòng qrels: {len(qrels)}")
print(f"lớp    : {pd.Series(QUERY_CLASS).value_counts().to_dict()}")

chunks : 4,133 · 179 bài
queries: 12 · số dòng qrels: 31
lớp    : {'literal': 4, 'paraphrase': 4, 'multi-doc': 4}


In [2]:
# ── BỐN HÀM BẠN CẦN VIẾT ───────────────────────────────────────
# Quy ước: `ranked_docs` là list doc_id đã xếp hạng (giảm dần theo điểm).
# Nếu không có kết quả nào relevant -> trả về 0.0 (không phải None, không lỗi).

def recall_at_k(ranked_docs, gold, k):
    """Tỉ lệ bài gold tìm được trong top-k.
    Công thức: (số bài gold nằm trong k vị trí đầu) / (tổng số bài gold)
    """
    ...  # TODO
    if len(gold) == 0:
        return 0.0
        
    count = 0
    for doc in ranked_docs[:k]:
        if doc in gold:
            count += 1
    return count / len(gold)

def precision_at_k(ranked_docs, gold, k):
    """Tỉ lệ vị trí trong top-k là bài gold.
    Công thức: (số bài gold nằm trong k vị trí đầu) / k
    """
    ...  # TODO
    if len(gold) == 0:
        return 0.0
    count = 0
    for doc in ranked_docs[:k]:
        if doc in gold:
            count += 1
    return count / k

def mrr(ranked_docs, gold):
    """Mean Reciprocal Rank cho MỘT query: nghịch đảo vị trí của bài gold ĐẦU TIÊN.
    Ví dụ: gold xuất hiện ở vị trí 1 -> 1.0 ; vị trí 2 -> 0.5 ; vị trí 5 -> 0.2
    Không tìm thấy bài gold nào -> 0.0
    """
    ...  # TODO

    for i, doc in enumerate(ranked_docs):
        if doc in gold:
            return 1 / (i + 1)

    return 0.0
    
def ndcg_at_k(ranked_docs, gold_graded, k):
    """nDCG@k với relevance có trọng số (0/1/2).
    DCG@k  = tổng qua từng vị trí i (bắt đầu từ 1): relevance / log2(i + 1)
    IDCG@k = DCG của thứ tự LÝ TƯỞNG (gold xếp theo điểm giảm dần)
    nDCG   = DCG / IDCG     (nếu IDCG == 0 -> trả về 0.0)
    """
    ...  # TODO
    DCG = 0.0
    IDCG = 0.0
    
    for i, doc in enumerate(ranked_docs[:k], start=1):
        rel = gold_graded.get(doc, 0)
        DCG += rel / (math.log2(i + 1))
        
    for i, rel in enumerate(sorted(gold_graded.values(), reverse=True)[:k], start=1):
        IDCG += rel / (math.log2(i + 1))

    if IDCG == 0:
        return 0.0
    return DCG / IDCG

print("4 hàm đã khai báo — giờ điền phần TODO")

4 hàm đã khai báo — giờ điền phần TODO


## ✅ Self-test: chạy cell dưới để kiểm tra 4 hàm của bạn

Bộ dữ liệu tay: ranking `[C, A, B]` (C không liên quan), gold `{A: 2, B: 1}`, k = 3.

Cell này **không sửa hàm của bạn** — nó chỉ so sánh với kỳ vọng và in ❌ nếu lệch.
Ra ❌ thì sửa **hàm**, đừng sửa kỳ vọng 😄

In [3]:
# ── SELF-TEST (không cần sửa) ──────────────────────────────────
RANKED = ["C", "A", "B"]
GOLD_BIN = {"A", "B"}
GOLD_GRD = {"A": 2, "B": 1}
K = 3

EXPECTED = {"Recall@3": 1.0000, "Precision@3": 0.6667, "MRR": 0.5000, "nDCG@3": 0.6697}

got = {}
try:
    got["Recall@3"] = recall_at_k(RANKED, GOLD_BIN, K)
    got["Precision@3"] = precision_at_k(RANKED, GOLD_BIN, K)
    got["MRR"] = mrr(RANKED, GOLD_BIN)
    got["nDCG@3"] = ndcg_at_k(RANKED, GOLD_GRD, K)
except Exception as e:
    print(f"⚠️  Có hàm chưa viết xong: {type(e).__name__}: {e}")

for name, want in EXPECTED.items():
    if name not in got:
        print(f"⏳ {name:12s} chưa chạy được")
        continue
    v = got[name]
    print(f"{'✅' if abs(v - want) < 0.001 else '❌'} {name:12s} của bạn = {v:.4f}   |   kỳ vọng = {want:.4f}")

# kiểm tra thêm 2 ca biên
if "MRR" in got:
    print(f"\nca biên — không có bài gold nào:")
    print(f"   mrr(['X','Y'], {{'A'}})      = {mrr(['X','Y'], {'A'})}   (kỳ vọng 0.0)")
    print(f"   recall_at_k(['X','Y'], {{'A'}}, 2) = {recall_at_k(['X','Y'], {'A'}, 2)}   (kỳ vọng 0.0)")
    print(f"   precision_at_k([], {{'A'}}, 2)     = {precision_at_k([], {'A'}, 2)}   (kỳ vọng 0.0)")

✅ Recall@3     của bạn = 1.0000   |   kỳ vọng = 1.0000
✅ Precision@3  của bạn = 0.6667   |   kỳ vọng = 0.6667
✅ MRR          của bạn = 0.5000   |   kỳ vọng = 0.5000
✅ nDCG@3       của bạn = 0.6697   |   kỳ vọng = 0.6697

ca biên — không có bài gold nào:
   mrr(['X','Y'], {'A'})      = 0.0   (kỳ vọng 0.0)
   recall_at_k(['X','Y'], {'A'}, 2) = 0.0   (kỳ vọng 0.0)
   precision_at_k([], {'A'}, 2)     = 0.0   (kỳ vọng 0.0)


# 🎯 PHẦN B

In [4]:
# ── Setup Phần B ───────────────────────────────────────────────
import sys
import numpy as np
from rank_bm25 import BM25Okapi

sys.path.insert(0, str(ROOT / "scripts"))
from explore import tokenize          # dùng CHUNG tokenizer với công cụ tra cứu

TEXTS = chunks["text"].tolist()        # 4.133 chunk
print(f"{len(TEXTS):,} chunk sẵn sàng")
print("  token của chunk đầu:", tokenize(TEXTS[0])[:12])


4,133 chunk sẵn sàng
  token của chunk đầu: ['overviews', 'tinh', 'nang', 'tich', 'hop', 'tri', 'tue', 'nhan', 'tao', 'google', 'tim', 'kiem']


In [5]:
# ── BA HÀM BẠN CẦN VIẾT ────────────────────────────────────────

def build_index(texts):
    """Dựng chỉ mục BM25 từ list văn bản chunk.
    Mảnh 1: tokenize từng văn bản  -> list[list[str]]
    Mảnh 2: BM25Okapi(corpus_tokens)
    """

    corpus_tokens = [tokenize(text) for text in texts]
    bm25 = BM25Okapi(corpus_tokens)
    return bm25


def retrieve(query, k=10):
    """Truy vấn: trả về TOP-k CHUNK đã xếp hạng giảm dần theo điểm.

    HỢP ĐỒNG ĐẦU RA (đúng thì cell sanity check mới chạy):
      list dict: {"chunk_id": str, "doc_id": str, "score": float, "text": str}

    Mảnh 1: q_tokens = tokenize(query)
    Mảnh 2: scores = BM25.get_scores(q_tokens)     -> điểm cho MỌI chunk
    Mảnh 3: lấy k chỉ số điểm cao nhất (gợi ý: np.argsort(scores)[::-1][:k])
    Mảnh 4: map chỉ số -> dòng trong `chunks` để lấy chunk_id / doc_id / text
    """
    q_tokens = tokenize(query)
    scores = BM25.get_scores(q_tokens)
    top_k_indices = np.argsort(scores)[::-1][:k]
    
    results = []
    for idx in top_k_indices:
        results.append({
            "chunk_id": chunks.iloc[idx]["chunk_id"],
            "doc_id": chunks.iloc[idx]["doc_id"],
            "score": scores[idx],
            "text": chunks.iloc[idx]["text"]
        })
    
    return results


def docs_from_chunks(hits, k=10):
    """Gộp danh sách CHUNK đã xếp hạng thành danh sách DOC đã xếp hạng.

    ⚠️ PHẦN TƯ DUY — bạn tự quyết và giải thích được:
       Điểm của một BÀI là gì khi bài đó có nhiều chunk?
         (a) MAX điểm chunk   (b) TỔNG   (c) ĐẾM số chunk khớp   (d) TRUNG BÌNH
       Gợi ý: chỉ cần MỘT đoạn của bài chứa câu trả lời là bài đó liên quan.

    Yêu cầu: mỗi doc_id xuất hiện ĐÚNG 1 LẦN, sắp giảm dần, cắt còn k.
    """
    doc_scores = {}
    for hit in hits:
        doc_id = hit["doc_id"]
        score = hit["score"]
        if doc_id not in doc_scores or score > doc_scores[doc_id]:
            doc_scores[doc_id] = score

    sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, score in sorted_docs[:k]]


print("3 hàm đã khai báo — giờ điền TODO")


3 hàm đã khai báo — giờ điền TODO


In [6]:
# ── Sanity check Phần B ────────────────────────────────────────
BM25 = build_index(TEXTS)

CHECKS = [
    ("q001", {"24345", "19893713"}),
    ("q002", {"30712", "20023680"}),
    ("q005", {"16665860", "24278"}),
]

for qid, gold in CHECKS:
    hits = retrieve(QUERIES[qid], k=10)
    docs = docs_from_chunks(hits, k=5)
    found = gold & set(docs)
    print(f"\n=== {qid} · lớp {QUERY_CLASS[qid]} ===")
    print(f"  query      : {QUERIES[qid][:70]}")
    print(f"  gold       : {sorted(gold)}")
    print(f"  top-3 chunk: {[(h['doc_id'], round(float(h['score']), 1)) for h in hits[:3]]}")
    print(f"  top-5 bài  : {docs}")
    print(f"  → tìm được {len(found)}/{len(gold)} bài gold trong top-5")



=== q001 · lớp literal ===
  query      : Học không có giám sát là gì?
  gold       : ['19893713', '24345']
  top-3 chunk: [('19895408', 8.2), ('24345', 8.2), ('19893713', 8.0)]
  top-5 bài  : ['19895408', '24345', '19893713', '3339820', '16760383']
  → tìm được 2/2 bài gold trong top-5

=== q002 · lớp literal ===
  query      : Phân loại nhị phân là gì?
  gold       : ['20023680', '30712']
  top-3 chunk: [('20023680', 9.5), ('20023680', 9.4), ('30712', 9.2)]
  top-5 bài  : ['20023680', '30712', '19991608', '20015375', '667810']
  → tìm được 2/2 bài gold trong top-5

=== q005 · lớp paraphrase ===
  query      : Vì sao mô hình đạt kết quả rất tốt trên dữ liệu huấn luyện nhưng lại k
  gold       : ['16665860', '24278']
  top-3 chunk: [('24278', 25.8), ('20023678', 24.5), ('20002585', 24.3)]
  top-5 bài  : ['24278', '20023678', '20002585', '14920124', '20022788']
  → tìm được 1/2 bài gold trong top-5


# 🎯 PHẦN C: Baseline + BM25 ranking


In [7]:
# ── PHẦN C: eval toàn bộ 12 query ──────────────────────────────

def bm25_rank(query, k=10, n_chunks=50):
    """Adapter BM25 -> danh sách doc_id đã xếp hạng.

    ⚠️ Lấy SÂU 50 chunk rồi mới gộp về k bài (bài học độ sâu retrieval:
       lấy 10 chunk thì chỉ được 6-8 bài, không bao giờ chạm tới hạng 10).
    Mảnh: hits = retrieve(query, k=n_chunks)
          return docs_from_chunks(hits, k=k)
    """
    hits = retrieve(query, k=n_chunks)
    return docs_from_chunks(hits, k)    


def evaluate(rank_fn, k=10):
    """Chạy rank_fn trên TOÀN BỘ query -> DataFrame 1 dòng/query.

    Cột: query_id, query_class, recall, precision, mrr, ndcg

    Mảnh 1: for qid, q in QUERIES.items():
                ranked = rank_fn(q, k=k)
    Mảnh 2: tính 4 metric với gold của query đó:
                recall_at_k(ranked, GOLD_BINARY[qid], k)
                precision_at_k(ranked, GOLD_BINARY[qid], k)
                mrr(ranked, GOLD_BINARY[qid])
                ndcg_at_k(ranked, GOLD_GRADED[qid], k)
    Mảnh 3: gom dict vào list -> pd.DataFrame(rows)
    """
    rows = []
    for qid, q in QUERIES.items():
        ranked = rank_fn(q, k=k)
        rows.append({
            'query_id': qid,
            'query_class': QUERY_CLASS[qid],
            'recall': recall_at_k(ranked, GOLD_BINARY[qid], k),
            'precision': precision_at_k(ranked, GOLD_BINARY[qid], k),
            'mrr': mrr(ranked, GOLD_BINARY[qid]),
            'ndcg': ndcg_at_k(ranked, GOLD_GRADED[qid], k),
        })
    
    return pd.DataFrame(rows)


In [8]:
# ── Báo cáo (không cần sửa) ────────────────────────────────────
COLS = ["recall", "precision", "mrr", "ndcg"]

def report(df, name, k=10):
    print(f"\n{'='*70}\n{name}   (k={k})\n{'='*70}")
    print("\n-- Trung bình theo lớp --")
    print(df.groupby("query_class")[COLS].mean().round(3).to_string())
    print("\n-- Tất cả --")
    print(df[COLS].mean().round(3).to_string())
    print("\n-- Chi tiết từng query --")
    print(df[["query_id", "query_class"] + COLS].round(3).to_string(index=False))

RESULTS_BM25 = evaluate(bm25_rank, k=10)
report(RESULTS_BM25, "BM25")



BM25   (k=10)

-- Trung bình theo lớp --
             recall  precision    mrr   ndcg
query_class                                 
literal       1.000      0.150  0.875  0.882
multi-doc     1.000      0.275  0.750  0.758
paraphrase    0.625      0.125  0.528  0.398

-- Tất cả --
recall       0.875
precision    0.183
mrr          0.718
ndcg         0.680

-- Chi tiết từng query --
query_id query_class  recall  precision   mrr  ndcg
    q001     literal     1.0        0.2 0.500 0.670
    q002     literal     1.0        0.2 1.000 0.860
    q003     literal     1.0        0.1 1.000 1.000
    q004     literal     1.0        0.1 1.000 1.000
    q005  paraphrase     0.5        0.1 1.000 0.380
    q006  paraphrase     0.0        0.0 0.000 0.000
    q007  paraphrase     1.0        0.2 0.111 0.339
    q008  paraphrase     1.0        0.2 1.000 0.875
    q009   multi-doc     1.0        0.3 0.500 0.648
    q010   multi-doc     1.0        0.2 1.000 0.832
    q011   multi-doc     1.0        0.3 1.00

In [9]:
bm25_rank("q005")

['19978788']

In [10]:
# ── PHẦN D: baseline ngẫu nhiên ────────────────────────────────
import random

ALL_DOC_IDS = sorted(chunks["doc_id"].unique())      # 179 bài

def random_rank(query, k=10, seed=42):
    """Baseline: xếp hạng bài NGẪU NHIÊN. Phải cố định seed để TÁI LẬP được.

    ⚠️ Cẩn thận: seed phải PHỤ THUỘC query, nếu không mọi query sẽ nhận
       cùng một thứ hạng (vô nghĩa).
    Gợi ý: random.Random(f"{seed}-{query}").sample(ALL_DOC_IDS, k)
    """
    rng = random.Random(f"{seed}-{query}")
    return rng.sample(ALL_DOC_IDS, k)   

RESULTS_RANDOM = evaluate(random_rank, k=10)
report(RESULTS_RANDOM, "NGẪU NHIÊN (baseline)")



NGẪU NHIÊN (baseline)   (k=10)

-- Trung bình theo lớp --
             recall  precision    mrr  ndcg
query_class                                
literal       0.000      0.000  0.000  0.00
multi-doc     0.000      0.000  0.000  0.00
paraphrase    0.125      0.025  0.031  0.03

-- Tất cả --
recall       0.042
precision    0.008
mrr          0.010
ndcg         0.010

-- Chi tiết từng query --
query_id query_class  recall  precision   mrr  ndcg
    q001     literal     0.0        0.0 0.000  0.00
    q002     literal     0.0        0.0 0.000  0.00
    q003     literal     0.0        0.0 0.000  0.00
    q004     literal     0.0        0.0 0.000  0.00
    q005  paraphrase     0.5        0.1 0.125  0.12
    q006  paraphrase     0.0        0.0 0.000  0.00
    q007  paraphrase     0.0        0.0 0.000  0.00
    q008  paraphrase     0.0        0.0 0.000  0.00
    q009   multi-doc     0.0        0.0 0.000  0.00
    q010   multi-doc     0.0        0.0 0.000  0.00
    q011   multi-doc     0.0    

# Phần D: Dense retrieval

In [11]:
# ── PHẦN D: dense retrieval ────────────────────────────────────
import os
# ⚠️ PHẢI set TRƯỚC khi import sentence_transformers
# (thư mục cache mặc định trên C: bị chặn quyền)
os.environ["HF_HOME"] = r"D:\Hermes-Workspace\hf-cache"
os.environ["HF_HUB_CACHE"] = r"D:\Hermes-Workspace\hf-cache\hub"

import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)
EMB = np.load(ROOT / "data" / "processed" / "embeddings_minilm.npy")
print("EMB:", EMB.shape)


def dense_rank(query, k=10, n_chunks=50):
    """Xếp hạng bằng embedding — CÙNG interface với bm25_rank.

    Mảnh 1: q_vec = model.encode([query], normalize_embeddings=True)[0]
    Mảnh 2: scores = EMB @ q_vec
            (4133×384 · 384 → 4133 điểm; hai bên đã normalize nên tích vô
             hướng chính là cosine similarity)
    Mảnh 3: lấy top n_chunks → gộp về k bài (dùng lại docs_from_chunks)
    """
    q_vec = model.encode([query], normalize_embeddings=True)[0]
    scores = EMB @ q_vec
    top_indices = np.argsort(scores)[::-1][:n_chunks]
    hits = []

    hits = [
        {
            "chunk_id": chunks.iloc[idx]["chunk_id"],
            "doc_id": chunks.iloc[idx]["doc_id"],
            "score": scores[idx],
            "text": chunks.iloc[idx]["text"],
        }
        for idx in top_indices
    ]
    
    return docs_from_chunks(hits, k)


RESULTS_DENSE = evaluate(dense_rank, k=10)
report(RESULTS_DENSE, "DENSE — MiniLM multilingual")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

EMB: (4133, 384)

DENSE — MiniLM multilingual   (k=10)

-- Trung bình theo lớp --
             recall  precision    mrr   ndcg
query_class                                 
literal       0.750      0.125  0.750  0.738
multi-doc     0.833      0.225  0.750  0.668
paraphrase    0.875      0.150  0.362  0.432

-- Tất cả --
recall       0.819
precision    0.167
mrr          0.621
ndcg         0.613

-- Chi tiết từng query --
query_id query_class  recall  precision  mrr  ndcg
    q001     literal   1.000        0.2 1.00 1.000
    q002     literal   1.000        0.2 1.00 0.950
    q003     literal   0.000        0.0 0.00 0.000
    q004     literal   1.000        0.1 1.00 1.000
    q005  paraphrase   0.500        0.1 0.50 0.240
    q006  paraphrase   1.000        0.1 0.25 0.431
    q007  paraphrase   1.000        0.2 0.50 0.627
    q008  paraphrase   1.000        0.2 0.20 0.429
    q009   multi-doc   0.667        0.2 1.00 0.634
    q010   multi-doc   1.000        0.2 0.50 0.693
    q011   mult

In [12]:
COMPARE = pd.DataFrame({
    "Random": RESULTS_RANDOM[COLS].mean(),
    "BM25": RESULTS_BM25[COLS].mean(),
    "Dense (MiniLM)": RESULTS_DENSE[COLS].mean(),
}).T.round(3)
COMPARE


,recall,precision,mrr,ndcg
Random,0.042,0.008,0.010,0.010
BM25,0.875,0.183,0.718,0.680
Dense (MiniLM),0.819,0.167,0.621,0.613


# PHẦN E: Hybrid (RRF)

In [13]:
# ── PHẦN E: Hybrid (RRF) ───────────────────────────────────────
# Lấy danh sách BÀI sâu từ mỗi retriever (50 chunk -> gộp thành danh sách bài)
def bm25_docs(query, depth=50):
    return bm25_rank(query, k=depth, n_chunks=depth)

def dense_docs(query, depth=50):
    return dense_rank(query, k=depth, n_chunks=depth)


In [14]:
def rrf_fuse(ranked_lists, K=60):
    """Gộp nhiều danh sách xếp hạng bằng Reciprocal Rank Fusion.

    Công thức:  score(doc) = Σ ( 1 / (K + rank) )   với rank bắt đầu từ 1

    ❓ VÌ SAO dùng THỨ HẠNG mà không dùng điểm số?
       Vì điểm BM25 (~0-30) và cosine (~0-1) KHÔNG cùng thang đo — cộng trực tiếp
       là vô nghĩa. RRF chỉ dùng thứ hạng, mà thứ hạng thì luôn so sánh được.
       Đây chính là lý do RRF được dùng rộng rãi: đơn giản và không cần chuẩn hoá.

    Mảnh 1: duyệt từng danh sách trong ranked_lists
            enumerate(______, start=______)  -> lấy (rank, doc_id)
    Mảnh 2: cộng dồn vào dict:  scores[doc] += 1 / (K + rank)
            (bài xuất hiện ở NHIỀU danh sách sẽ được cộng nhiều lần — đó là điểm mấu chốt)
    Mảnh 3: sắp giảm dần theo điểm, trả về list doc_id
    """
    scores = {}
    for ranked_list in ranked_lists:
        for rank, doc_id in enumerate(ranked_list, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (K + rank)

    sorted_scores = sorted(scores.items(), key = lambda x: x[1], reverse= True)
    return [doc_id for doc_id, score in sorted_scores]

def hybrid_rank(query, k=10, depth=50):
    """Gộp BM25 + dense ở mức BÀI, rồi cắt còn k bài.

    Mảnh: lists = [bm25_docs(query, depth), dense_docs(query, depth)]
          return rrf_fuse(lists, K=60)[:k]
    """
    lists = [
        bm25_docs(query, depth), 
        dense_docs(query, depth),
    ]

    return rrf_fuse(lists, K=60)[:k]


RESULTS_HYBRID = evaluate(hybrid_rank, k=10)
report(RESULTS_HYBRID, "HYBRID (BM25 + Dense, RRF)")



HYBRID (BM25 + Dense, RRF)   (k=10)

-- Trung bình theo lớp --
             recall  precision    mrr   ndcg
query_class                                 
literal       1.000      0.150  0.786  0.821
multi-doc     1.000      0.275  1.000  0.877
paraphrase    0.875      0.150  0.667  0.591

-- Tất cả --
recall       0.958
precision    0.192
mrr          0.817
ndcg         0.763

-- Chi tiết từng query --
query_id query_class  recall  precision   mrr  ndcg
    q001     literal     1.0        0.2 1.000 0.950
    q002     literal     1.0        0.2 1.000 1.000
    q003     literal     1.0        0.1 0.143 0.333
    q004     literal     1.0        0.1 1.000 1.000
    q005  paraphrase     0.5        0.1 1.000 0.380
    q006  paraphrase     1.0        0.1 0.167 0.356
    q007  paraphrase     1.0        0.2 1.000 1.000
    q008  paraphrase     1.0        0.2 0.500 0.627
    q009   multi-doc     1.0        0.3 1.000 0.852
    q010   multi-doc     1.0        0.2 1.000 0.920
    q011   multi-doc  

In [17]:
for K in (10, 30, 60, 100):
    fn = lambda q, k, K=K: rrf_fuse([bm25_docs(q, 50), dense_docs(q, 50)], K=K)[:k]
    m = evaluate(fn, k=10)[COLS].mean().round(3)
    print(f"RRF K={K:3d} → recall {m['recall']:.3f} · MRR {m['mrr']:.3f} · nDCG {m['ndcg']:.3f}")

RRF K= 10 → recall 0.958 · MRR 0.829 · nDCG 0.770
RRF K= 30 → recall 0.958 · MRR 0.817 · nDCG 0.762
RRF K= 60 → recall 0.958 · MRR 0.817 · nDCG 0.763
RRF K=100 → recall 0.958 · MRR 0.817 · nDCG 0.763
